1. Thread() — basic threads
2. start() and join()
3. Multiple threads with a loop
4. ThreadPoolExecutor
5. Shared variables
6. Lock
7. Producer–consumer with Queue
8. Multithreading vs multiprocessing
9. Multithreading vs asyncio

# 1. Thread() — basic threads
# 2. start() and join()

In [1]:
import threading
import time

def task(name):
    print(f"{name} started")
    time.sleep(2)
    print(f"{name} completed")

t1 = threading.Thread(target=task, args=('task 1',))
t2 = threading.Thread(target=task, args=('task 2',))
t3 = threading.Thread(target=task, args=('task 3',))

t1.start()
t2.start()
t3.start()

t1.join()
t2.join()
t3.join()
print("all tasks completed")


task 1 started
task 2 started
task 3 started
task 1 completed
task 2 completed
task 3 completed
all tasks completed


# 3. Multiple threads with a loop

In [2]:


import threading
import time


def process_number(number):
    print(f"Processing {number}")
    time.sleep(1)
    print(f"Completed {number}")


threads = []

for i in range(1, 6):
    thread = threading.Thread(
        target=process_number,
        args=(i,)
    )

    threads.append(thread)
    thread.start()


for thread in threads:
    thread.join()

print("Done")

Processing 1
Processing 2
Processing 3
Processing 4
Processing 5
Completed 1
Completed 2
Completed 3
Completed 4
Completed 5
Done


# 4. Practical Example — Download/API Simulation

In [3]:
import threading
import time


def fetch_data(api_name):
    print(f"Calling {api_name}")

    # Simulate API/network delay
    time.sleep(2)

    print(f"Response received from {api_name}")


apis = [
    "Users API",
    "Orders API",
    "Products API",
    "Payments API"
]

threads = []

start = time.time()

for api in apis:
    thread = threading.Thread(
        target=fetch_data,
        args=(api,)
    )

    threads.append(thread)
    thread.start()


for thread in threads:
    thread.join()


end = time.time()

print(f"Total time: {end - start:.2f} seconds")

Calling Users API
Calling Orders API
Calling Products API
Calling Payments API
Response received from Users APIResponse received from Orders API

Response received from Payments API
Response received from Products API
Total time: 2.02 seconds


# 4. ThreadPoolExecutor

## 4.1. Using ThreadPoolExecutor (File prcessing)

In [4]:
from concurrent.futures import ThreadPoolExecutor
import time

def process_file(file):
    print(f"Processing {file}")

    time.sleep(2)

    return f"\n{file} completed"

files = [
    "file1.csv",
    "file2.csv",
    "file3.csv",
    "file4.csv",
    "file5.csv"
]

with ThreadPoolExecutor(max_workers=2) as executor:
    results = executor.map(process_file, files)
    for result in results:
        print(result)


Processing file1.csv
Processing file2.csv
Processing file3.csv
file1.csv completed

Processing file4.csv

file2.csv completed
Processing file5.csv
file3.csv completed


file4.csv completed

file5.csv completed


## 4.2. ThreadPoolExecutor

In [5]:
from concurrent.futures import ThreadPoolExecutor
import time


def download_file(file):
    print(f"Downloading {file}")

    time.sleep(2)

    return f"\n{file} downloaded"


files = [
    "file1.csv",
    "file2.csv",
    "file3.csv",
    "file4.csv"
]


with ThreadPoolExecutor(max_workers=2) as executor:

    futures = []

    for file in files:
        future = executor.submit(download_file, file)
        futures.append(future)

    for future in futures:
        print(future.result())

file1.csv downloaded


file2.csv downloaded

file3.csv downloaded

file4.csv downloaded


## Thread Pool vs Creating Threads Manually

| Manual Threads | Thread Pool |
|---|---|
| Create threads yourself | Pool manages threads |
| Need `start()` / `join()` | Pool manages lifecycle |
| More code | Less code |
| Harder to control many tasks | Easy to control |
| Good for small/simple cases | Better for many similar tasks |
| `threading.Thread` | `ThreadPoolExecutor` |

# 4.3 Read multiple files using ThreadPoolExecutor

In [6]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import time


def read_file(file_path):
    """
    Read a single file and return its content.
    """

    print(f"Reading: {file_path}")

    try:
        with open(file_path, "r", encoding="utf-8") as file:
            content = file.read()

        print(f"Completed: {file_path}")

        return {
            "file": file_path,
            "status": "success",
            "content": content
        }

    except Exception as e:
        print(f"Error reading {file_path}: {e}")

        return {
            "file": file_path,
            "status": "failed",
            "content": None
        }


def main():

    files = [
        "data/file1.txt",
        "data/file2.txt",
        "data/file3.txt",
        "data/file4.txt",
        "data/file5.txt"
    ]

    start_time = time.time()

    # Create a thread pool
    with ThreadPoolExecutor(max_workers=3) as executor:

        # Submit all files to the thread pool
        futures = [
            executor.submit(read_file, file)
            for file in files
        ]

        # Get results
        results = []

        for future in futures:
            result = future.result()
            results.append(result)

    end_time = time.time()

    print("\n========== RESULTS ==========")

    for result in results:

        print(f"\nFile: {result['file']}")
        print(f"Status: {result['status']}")

        if result["content"] is not None:
            print("Content:")
            print(result["content"])

    print(f"\nTotal time: {end_time - start_time:.2f} seconds")


if __name__ == "__main__":
    main()

Reading: data/file1.txt
Reading: data/file2.txt
Error reading data/file1.txt: [Errno 2] No such file or directory: 'data/file1.txt'
Reading: data/file3.txt
Error reading data/file2.txt: [Errno 2] No such file or directory: 'data/file2.txt'
Error reading data/file3.txt: [Errno 2] No such file or directory: 'data/file3.txt'
Reading: data/file4.txt
Reading: data/file5.txt
Error reading data/file4.txt: [Errno 2] No such file or directory: 'data/file4.txt'
Error reading data/file5.txt: [Errno 2] No such file or directory: 'data/file5.txt'

========== RESULTS ==========

File: data/file1.txt
Status: failed

File: data/file2.txt
Status: failed

File: data/file3.txt
Status: failed

File: data/file4.txt
Status: failed

File: data/file5.txt
Status: failed

Total time: 0.00 seconds


# 5. Interview Practice — Shared Counter

In [7]:

import threading

counter = 0


def increment():
    global counter

    for _ in range(100000):
        counter += 1


t1 = threading.Thread(target=increment)
t2 = threading.Thread(target=increment)

t1.start()
t2.start()

t1.join()
t2.join()

print(counter)

200000


# 6. Lock
## prevent a race condition.

In [8]:

import threading

counter = 0

lock = threading.Lock()

def increment():
    global counter
    for _ in range(100000):
        with lock:
            counter += 1

def increment_1():
    global counter

    for _ in range(100000):
        lock.acquire()
        try:
            counter += 1
        finally:
            lock.release()

t1 = threading.Thread(target=increment)
t2 = threading.Thread(target=increment)

t1.start()
t2.start()

t1.join()
t2.join()

print(counter)

200000


# 7. Producer–consumer with Queue

What is Producer-Consumer pattern?
The Producer-Consumer pattern is a multithreading design where producer threads generate tasks or data and put them into a thread-safe queue. 
Consumer threads retrieve items from the queue and process them. 
Python's queue.Queue provides thread-safe communication between the threads and helps avoid race conditions.


In [9]:
import threading
import queue
import time

q = queue.Queue()


def producer(name):
    for i in range(5):
        item = f"{name}-{i}"
        q.put(item)
        print(f"{name} produced {item}")
        time.sleep(0.5)


def consumer(name):
    while True:
        item = q.get()

        if item is None:
            q.task_done()
            break

        print(f"{name} processing {item}")
        time.sleep(1)

        q.task_done()


producers = [
    threading.Thread(target=producer, args=("Producer-1",)),
    threading.Thread(target=producer, args=("Producer-2",))
]

consumers = [
    threading.Thread(target=consumer, args=("Consumer-1",)),
    threading.Thread(target=consumer, args=("Consumer-2",))
]

for t in producers:
    t.start()

for t in consumers:
    t.start()

for t in producers:
    t.join()

# Stop consumers
for _ in consumers:
    q.put(None)

for t in consumers:
    t.join()

print("All work completed")

Producer-1 produced Producer-1-0
Producer-2 produced Producer-2-0
Consumer-1 processing Producer-1-0
Consumer-2 processing Producer-2-0
Producer-2 produced Producer-2-1
Producer-1 produced Producer-1-1
Consumer-1 processing Producer-2-1
Consumer-2 processing Producer-1-1
Producer-2 produced Producer-2-2
Producer-1 produced Producer-1-2
Producer-2 produced Producer-2-3Producer-1 produced Producer-1-3

Consumer-1 processing Producer-2-2Consumer-2 processing Producer-1-2

Producer-1 produced Producer-1-4
Producer-2 produced Producer-2-4
Consumer-1 processing Producer-2-3Consumer-2 processing Producer-1-3

Consumer-2 processing Producer-1-4Consumer-1 processing Producer-2-4

All work completed
